In [1]:
import os

print("Train folders:", os.listdir("train"))
print("Test folders:", os.listdir("test"))

Train folders: ['benign', 'malignant']
Test folders: ['benign', 'malignant']


In [2]:
!pip install tensorflow


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Parameters
IMG_SIZE = 224
BATCH = 32
EPOCHS = 15

# Data Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_gen = ImageDataGenerator(rescale=1./255)

# Load Data
train_data = train_gen.flow_from_directory(
    "train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="binary",
    subset="training"
)

val_data = train_gen.flow_from_directory(
    "train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="binary",
    subset="validation"
)

test_data = test_gen.flow_from_directory(
    "test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="binary"
)

# Base Model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# Custom Head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(base_model.input, output)

# Compile
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Train
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

# Evaluate
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc*100)

# Save Model
model.save("skin_cancer_model.keras")
print("Model Saved Successfully!")

Found 2110 images belonging to 2 classes.
Found 527 images belonging to 2 classes.
Found 660 images belonging to 2 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 155s 2s/step - accuracy: 0.7900 - loss: 0.4715 - val_accuracy: 0.7989 - val_loss: 0.4357
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 218s 3s/step - accuracy: 0.8303 - loss: 0.3635 - val_accuracy: 0.7742 - val_loss: 0.4584
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 136s 2s/step - accuracy: 0.8341 - loss: 0.3635 - val_accuracy: 0.7628 - val_loss: 0.4764
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.8422 - loss: 0.3363 - val_accuracy: 0.8065 - val_loss: 0.4279
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.8602 - loss: 0.3027 - val_accuracy: 0.7894 - val_loss: 0.4412
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 249s 4s/step - accuracy: 0.8588 - loss: 0.3079 - val_accuracy: 0.8027 - val_loss: 0.4191
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.8545 - loss: 0.3034 - val_accuracy: 0.8235 - val_loss: 0.4073
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.8664 - loss: 0.2954 - val_accuracy: 0.8102 - v

In [5]:
!pip install streamlit pillow tensorflow opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   - -------------------------------------- 1.6/40.2 MB 5.6 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/40.2 MB 5.8 MB/s eta 0:00:07
   --- ------------------------------------ 3.1/40.2 MB 4.5 MB/s eta 0:00:09
   --- ------------------------------------ 3.7/40.2 MB 4.0 MB/s eta 0:00:10
   ---- ----------------------------------- 4.7/40.2 MB 4.3 MB/s eta 0:00:09
   ----- ---------------------------------- 6.0/40.2 MB 4.6 MB/s eta 0:00:08
   ------- -------------------------------- 7.1/40.2 MB 4.7 MB/s eta 0:00:08
   -------- ------------------------------- 8.1/40.2 MB 4.7 MB/s eta 0:00:07
   -------- ------------------------------- 8.7/40.2 MB 4.4 MB/s eta 0:00:08
   -------- ------------------------------- 8.9/40.2 MB 4.0 MB/s eta 0:00:08
   --------- -------


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip
